In [ ]:
from typing import OrderedDict
from tqdm.notebook import tqdm
import torch
import datasets
import networks


dataset = datasets.LIGOLChirping(
    episode_duration_s=1.0,
    n_sources=1,
    sample_rate_Hz=10000,
    episodes_per_epoch=128,
)
loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=64,
    num_workers=8,
    pin_memory=True,
    prefetch_factor=8,
)

for data, labels in loader:
    print(f"{data.shape=} {data.dtype=}\n{labels.shape=} {labels.dtype=}")
    break

In [ ]:
n_params, n_sources = labels.shape[1:]


class ToyModel(torch.nn.Module):
    def __init__(self, n_params, n_sources):
        super().__init__()
        self.conv = networks.ConvNet2d(
            in_channels=1,
            hidden_channels=[16, 16, 16, 16, 16, 16, 16],
            out_channels=n_params * n_sources,
            kernel_size=(3, 7),
        )
        self.flatten = torch.nn.Flatten()
        self.linear = torch.nn.LazyLinear(n_params * n_sources)
        self.normalize = torch.nn.BatchNorm1d(n_params * n_sources)

    def forward(self, data, labels):
        output = self.linear(self.flatten(self.conv(data)))
        target = self.normalize(self.flatten(labels))
        return output, target


model = ToyModel(n_params=labels.shape[1], n_sources=labels.shape[2])
print(data.shape, model(data, labels)[0].shape)
print(model)

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
model.train()

losslog = []
for epoch in tqdm(range(100)):
    losslog.append([])
    for data, labels in loader:
        data, labels = data.to(device), labels.to(device)
        output, target = model(data, labels)
        loss = torch.nn.functional.smooth_l1_loss(output, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losslog[-1].append(loss)
    print(f"epoch {epoch} loss {torch.tensor(losslog[-1]).mean().item():.3f}")

In [ ]:
import matplotlib.pyplot as plt
plt.semilogy(torch.tensor(losslog).mean(dim=1).numpy())